1. Data cleaning: standardize date ranges and spatial units from each data source, handle facilities with poor or absent location information.


See `blockify_years.ipynb` for various cleaning tasks - notably, converting yearly data into five-year blocks for the crime rate, fine particles, median income, and infant mortality metrics. This notebook operates on the cleaned CSVs produced by that one.

All the metric tables consumed by this notebook should have the following columns:

- `TimePeriod`, holding the five-year block (in the format `YYYY-YY`, ie `2015-19`)
- `GeoID`, holding the community district ID number
- Some other column with the actual metric value for that time period and community district

The name of the metric-value column should be specified in the `metric_specs` list, along with the name to be used for that metric in the merged table.

Load the cleaned CSVs

In [1]:
import pandas as pd
import numpy as np

folder_path = "datasets/"
params_path = "runs/params/"
facs_path = "runs/facs/"

facilities = pd.read_csv(folder_path + "facilities.csv", low_memory=False)

# (file name, metric-value column in that file, name to use in the merged table)
metric_specs = [
    ("rent_burden.csv",           "Percent",                       "rent_burden_pct"),
    ("crowding.csv",              "Percent",                       "crowding_pct"),
    ("median_income_parsed.csv",  "median_income",                 "median_income"),
    ("poverty.csv",               "Percent",                       "poverty_pct"),
    ("crime_rate_parsed.csv",     "crime_all_rt",                  "crimes_per_1000_ppl"),
    ("high_school.csv",           "Percent",                       "high_school_grad_pct"),
    ("fine_particles_parsed.csv", "Annual mean mcg/m3",            "fine_particle_concentration"),
    ("walk_to_work.csv",          "Walked (percent)",              "walk_to_work_rate"),
    ("infant_mortality_parsed.csv", "infant_mortality",              "infant_mortality_per_1000"),
    ("indep_living.csv",          "Percent",                       "independency_pct"),
]
metric_cols = [spec[2] for spec in metric_specs]

metric_tables = {}
for file_name, value_col, new_name in metric_specs:
    df = pd.read_csv(folder_path + file_name, thousands=",")
    metric_tables[new_name] = df[["TimePeriod", "GeoID", value_col]].rename(columns={value_col: new_name})
    print(f"{file_name:28s} {len(df):4d} rows, periods {df['TimePeriod'].min()} .. {df['TimePeriod'].max()}")

rent_burden.csv               531 rows, periods 2007-11 .. 2015-19
crowding.csv                  531 rows, periods 2007-11 .. 2015-19
median_income_parsed.csv      944 rows, periods 2005-09 .. 2020-24
poverty.csv                   531 rows, periods 2007-11 .. 2015-19
crime_rate_parsed.csv         944 rows, periods 2006-10 .. 2021-25
high_school.csv               531 rows, periods 2007-11 .. 2015-19
fine_particles_parsed.csv     767 rows, periods 2009-13 .. 2021-25
walk_to_work.csv              531 rows, periods 2007-11 .. 2015-19
infant_mortality_parsed.csv  1003 rows, periods 2002-06 .. 2018-22
indep_living.csv              472 rows, periods 2008-12 .. 2015-19


**Facilities with missing location info.** 821 facilities have `cd = 0` (no community district). About 140 of those still have valid latitude/longitude, so we assign them a district with a KNN classifier trained on the facilities whose district is known (a facility most likely belongs to the same district as its nearest neighbours). Facilities with no coordinates (lat/lon = 0) are imputed from their zip code instead (most common district among facilities with that zip code); any that still can't be placed are dropped.

Codes 164, 226-228, 355-356, 480-484 and 595 are "joint interest areas" (Central Park, the airports, large parks, etc.), not community districts, so those facilities are left out of the district counts.

In [2]:
from sklearn.neighbors import KNeighborsClassifier

# the 59 community districts
geoids = [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112,
          201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212,
          301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318,
          401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414,
          501, 502, 503]

has_coords = (facilities["latitude"] != 0) & (facilities["longitude"] != 0)
known_cd = facilities["cd"].isin(geoids)
missing_cd = facilities["cd"] == 0

# KNN on coordinates for facilities that have a location but no district
knn_cd = KNeighborsClassifier(n_neighbors=5)
knn_cd.fit(facilities.loc[known_cd & has_coords, ["latitude", "longitude"]],
           facilities.loc[known_cd & has_coords, "cd"])
by_coords = missing_cd & has_coords
facilities.loc[by_coords, "cd"] = knn_cd.predict(facilities.loc[by_coords, ["latitude", "longitude"]])

# zip code fallback for facilities with no coordinates
zip_to_cd = facilities[known_cd].groupby("zipcode")["cd"].agg(lambda s: s.mode().iloc[0])
by_zip = (facilities["cd"] == 0) & facilities["zipcode"].notna()
facilities.loc[by_zip, "cd"] = facilities.loc[by_zip, "zipcode"].map(zip_to_cd).fillna(0)

print("assigned by coordinates:", by_coords.sum())
print("assigned by zip code:   ", (by_zip & (facilities["cd"] != 0)).sum())
print("still unplaced (dropped):", (facilities["cd"] == 0).sum())
print("in joint interest areas (excluded):", (~facilities["cd"].isin(geoids) & (facilities["cd"] != 0)).sum())

facilities = facilities[facilities["cd"].isin(geoids)].copy()
facilities["cd"] = facilities["cd"].astype(int)

assigned by coordinates: 143
assigned by zip code:    397
still unplaced (dropped): 281
in joint interest areas (excluded): 146




2. Data consolidation: combine information from a wide range of data sources to build a comprehensive picture of each NYC community district.




In [3]:
# join all the individual metric tables by TimePeriod and GeoID, using a left join onto rent burden
# so the time periods are rent burden's (2007-11 .. 2015-19); periods absent from a dataset become NaN
all_metrics = metric_tables["rent_burden_pct"]
for name in metric_cols[1:]:
    all_metrics = all_metrics.merge(metric_tables[name], on=["TimePeriod", "GeoID"], how="left")
all_metrics = all_metrics[all_metrics["GeoID"].isin(geoids)].reset_index(drop=True)

print(all_metrics.shape)
print("missing values per metric:")
print(all_metrics[metric_cols].isna().sum())
all_metrics.head(10)

(531, 12)
missing values per metric:
rent_burden_pct                  0
crowding_pct                     0
median_income                    0
poverty_pct                      0
crimes_per_1000_ppl              0
high_school_grad_pct             0
fine_particle_concentration    118
walk_to_work_rate                0
infant_mortality_per_1000        0
independency_pct                59
dtype: int64


,TimePeriod,GeoID,rent_burden_pct,crowding_pct,median_income,poverty_pct,crimes_per_1000_ppl,high_school_grad_pct,fine_particle_concentration,walk_to_work_rate,infant_mortality_per_1000,independency_pct
0,2015-19,201,58.4,12.7,30722.2,40.1,24.98,63.8,8.54,9.7,5.34,8.5
1,2015-19,202,58.4,12.7,30722.2,40.1,19.32,63.8,8.54,9.7,3.24,8.5
2,2015-19,203,58.4,12.3,31323.6,38.6,17.52,65.9,7.86,9.8,5.34,9.0
3,2015-19,204,59.1,17.6,38366.4,34.5,15.36,67.4,8.20,8.8,4.48,7.3
4,2015-19,205,61.7,17.8,35604.8,36.7,14.62,65.6,8.28,7.4,4.88,6.7
5,2015-19,206,58.4,12.3,31323.6,38.6,17.62,65.9,7.98,9.8,6.32,9.0
6,2015-19,207,60.7,18.5,43962.0,29.5,13.38,69.7,7.80,8.9,3.96,6.1
7,2015-19,208,52.4,8.7,72126.4,17.6,9.08,81.7,7.56,7.4,3.64,6.7
8,2015-19,209,55.4,11.7,50088.0,26.8,12.24,73.2,7.62,5.9,5.98,7.7
9,2015-19,210,45.8,5.1,71028.4,11.9,9.44,84.4,7.52,5.3,3.94,8.2


**KNN imputation.** None of the source CSVs have blank cells; the missing values come from datasets that don't cover every time period (fine particles start in 2009-13, independent living in 2008-12). Each missing value is filled in from the 5 most similar (district, time period) rows, where similarity is measured on the metrics that row *does* have. The metrics are standardized first so large-valued columns like median income don't dominate the distance.

In [4]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(all_metrics[metric_cols])  # NaNs are ignored while fitting

imputer = KNNImputer(n_neighbors=5, weights="distance")
all_metrics[metric_cols] = scaler.inverse_transform(imputer.fit_transform(scaled))

print("missing values after imputation:", all_metrics[metric_cols].isna().sum().sum())
all_metrics.head(10)

missing values after imputation: 0


,TimePeriod,GeoID,rent_burden_pct,crowding_pct,median_income,poverty_pct,crimes_per_1000_ppl,high_school_grad_pct,fine_particle_concentration,walk_to_work_rate,infant_mortality_per_1000,independency_pct
0,2015-19,201,58.4,12.7,30722.2,40.1,24.98,63.8,8.54,9.7,5.34,8.5
1,2015-19,202,58.4,12.7,30722.2,40.1,19.32,63.8,8.54,9.7,3.24,8.5
2,2015-19,203,58.4,12.3,31323.6,38.6,17.52,65.9,7.86,9.8,5.34,9.0
3,2015-19,204,59.1,17.6,38366.4,34.5,15.36,67.4,8.20,8.8,4.48,7.3
4,2015-19,205,61.7,17.8,35604.8,36.7,14.62,65.6,8.28,7.4,4.88,6.7
5,2015-19,206,58.4,12.3,31323.6,38.6,17.62,65.9,7.98,9.8,6.32,9.0
6,2015-19,207,60.7,18.5,43962.0,29.5,13.38,69.7,7.80,8.9,3.96,6.1
7,2015-19,208,52.4,8.7,72126.4,17.6,9.08,81.7,7.56,7.4,3.64,6.7
8,2015-19,209,55.4,11.7,50088.0,26.8,12.24,73.2,7.62,5.9,5.98,7.7
9,2015-19,210,45.8,5.1,71028.4,11.9,9.44,84.4,7.52,5.3,3.94,8.2


Calculate domain percentiles

In [5]:
# percentile of each metric among all districts in the same time period
# (same definition as Spark's percent_rank: (rank - 1) / (n - 1))
percentiles = all_metrics.copy()
for col in metric_cols:
    rank = percentiles.groupby("TimePeriod")[col].rank(method="min")
    n = percentiles.groupby("TimePeriod")[col].transform("count")
    percentiles[col + " percentile"] = (rank - 1) / (n - 1)

percentiles[percentiles["TimePeriod"] == "2015-19"].head(10)

,TimePeriod,GeoID,rent_burden_pct,crowding_pct,median_income,poverty_pct,crimes_per_1000_ppl,high_school_grad_pct,fine_particle_concentration,walk_to_work_rate,...,rent_burden_pct percentile,crowding_pct percentile,median_income percentile,poverty_pct percentile,crimes_per_1000_ppl percentile,high_school_grad_pct percentile,fine_particle_concentration percentile,walk_to_work_rate percentile,infant_mortality_per_1000 percentile,independency_pct percentile
0,2015-19,201,58.4,12.7,30722.2,40.1,24.98,63.8,8.54,9.7,...,0.810345,0.810345,0.000000,0.982759,0.948276,0.017241,0.810345,0.655172,0.741379,0.913793
1,2015-19,202,58.4,12.7,30722.2,40.1,19.32,63.8,8.54,9.7,...,0.810345,0.810345,0.000000,0.982759,0.913793,0.017241,0.810345,0.655172,0.379310,0.913793
2,2015-19,203,58.4,12.3,31323.6,38.6,17.52,65.9,7.86,9.8,...,0.810345,0.775862,0.034483,0.948276,0.810345,0.068966,0.603448,0.689655,0.741379,0.965517
3,2015-19,204,59.1,17.6,38366.4,34.5,15.36,67.4,8.20,8.8,...,0.896552,0.896552,0.103448,0.913793,0.758621,0.103448,0.724138,0.568966,0.603448,0.827586
4,2015-19,205,61.7,17.8,35604.8,36.7,14.62,65.6,8.28,7.4,...,0.982759,0.913793,0.086207,0.931034,0.741379,0.051724,0.741379,0.448276,0.706897,0.672414
5,2015-19,206,58.4,12.3,31323.6,38.6,17.62,65.9,7.98,9.8,...,0.810345,0.775862,0.034483,0.948276,0.827586,0.068966,0.689655,0.689655,0.931034,0.965517
6,2015-19,207,60.7,18.5,43962.0,29.5,13.38,69.7,7.80,8.9,...,0.965517,0.931034,0.137931,0.862069,0.655172,0.120690,0.568966,0.586207,0.517241,0.551724
7,2015-19,208,52.4,8.7,72126.4,17.6,9.08,81.7,7.56,7.4,...,0.603448,0.500000,0.517241,0.534483,0.327586,0.465517,0.448276,0.448276,0.448276,0.672414
8,2015-19,209,55.4,11.7,50088.0,26.8,12.24,73.2,7.62,5.9,...,0.741379,0.758621,0.189655,0.793103,0.568966,0.172414,0.482759,0.293103,0.879310,0.844828
9,2015-19,210,45.8,5.1,71028.4,11.9,9.44,84.4,7.52,5.3,...,0.241379,0.137931,0.500000,0.258621,0.413793,0.586207,0.413793,0.258621,0.500000,0.896552


Calculate overall quality metric

In [6]:
weights = {
    "rent_burden_pct":              -1,
    "crowding_pct":                 -1,
    "median_income":                 5,
    "poverty_pct":                  -2,
    "crimes_per_1000_ppl":          -1.3,
    "high_school_grad_pct":          4,
    "fine_particle_concentration": -0.5,
    "walk_to_work_rate":             3.4,
    "infant_mortality_per_1000":  -1,
    "independency_pct":              -1.3,
}
total_magnitude = sum(abs(w) for w in weights.values())

# OverallQuality ranges from 0.0 to 1.0; for negative-weight metrics a low percentile is good
overall_quality = percentiles[["GeoID", "TimePeriod"]].copy()
overall_quality["OverallQuality"] = 0.0
for col, w in weights.items():
    pct = percentiles[col + " percentile"]
    overall_quality["OverallQuality"] += (abs(w) / total_magnitude) * ((1.0 - pct) if w < 0 else pct)

overall_quality.sort_values("OverallQuality", ascending=False).head(10)

,GeoID,TimePeriod,OverallQuality
273,108,2011-15,0.932212
214,108,2012-16,0.928932
332,108,2010-14,0.926241
155,108,2013-17,0.923633
450,108,2008-12,0.922960
391,108,2009-13,0.921447
96,108,2014-18,0.921278
509,108,2007-11,0.920521
330,106,2010-14,0.910093
389,106,2009-13,0.906224


Add facility counts columns to make training table

In [7]:
facility_counts = (
    pd.crosstab(facilities["cd"], facilities["facgroup"])
    .reindex(geoids, fill_value=0)
    .rename_axis("GeoID")
    .rename_axis(None, axis=1)
    .reset_index()
)
facility_types = [c for c in facility_counts.columns if c != "GeoID"]

trainer = overall_quality.merge(facility_counts, on="GeoID", how="left")
trainer.to_csv(folder_path + "trainer.csv", index=False)

print(trainer.shape)
trainer.head(10)

(531, 28)


,GeoID,TimePeriod,OverallQuality,ADULT SERVICES,CAMPS,CHILD SERVICES AND WELFARE,"CITY AGENCY PARKING, MAINTENANCE, AND STORAGE",CULTURAL INSTITUTIONS,DAY CARE AND PRE-KINDERGARTEN,EMERGENCY SERVICES,...,OTHER PROPERTY,PARKS AND PLAZAS,PUBLIC SAFETY,SCHOOLS (K-12),SOLID WASTE,TELECOMMUNICATIONS,TRANSPORTATION,VOCATIONAL AND PROPRIETARY SCHOOLS,WATER AND WASTEWATER,YOUTH SERVICES
0,201,2015-19,0.158200,9,1,54,11,15,54,8,...,1,46,4,86,35,1,75,12,1,154
1,202,2015-19,0.178049,1,0,23,6,6,23,1,...,0,34,2,32,47,0,31,1,6,61
2,203,2015-19,0.197981,1,1,49,6,4,59,6,...,1,44,1,62,14,0,22,1,0,108
3,204,2015-19,0.210597,3,2,40,17,10,56,4,...,0,47,2,69,23,0,55,0,0,128
4,205,2015-19,0.175021,2,0,26,3,7,45,4,...,0,40,1,39,14,0,45,8,0,84
5,206,2015-19,0.185534,4,0,31,10,6,52,4,...,0,45,2,58,25,0,52,3,0,84
6,207,2015-19,0.257275,1,0,27,6,14,50,3,...,0,23,1,42,19,0,41,6,1,61
7,208,2015-19,0.484272,2,10,20,0,10,65,2,...,0,31,1,48,13,0,23,1,3,55
8,209,2015-19,0.228764,0,1,36,11,8,51,3,...,1,47,1,63,34,0,47,1,4,95
9,210,2015-19,0.513036,0,1,20,3,9,48,4,...,0,26,3,48,20,0,41,2,2,53


3. Model training: train a machine learning model on the raw scores for the quality of life domains of each district, so that the model determines how the count of a facility type in a district may affect the raw scores of domains for that district.


XGBoost regressor: facility counts → `OverallQuality`.

The facilities table is a single snapshot, so a district has the same facility counts in every time period; only the target changes across periods. We train on every (district, period) row, which effectively teaches the model each district's typical score while smoothing out year-to-year noise. To get an honest accuracy estimate, cross-validation is grouped by district so a district never appears in both the training and the test fold.

Each run appends its stats (CV R², MAE, baseline, settings) to `runs/params/model_runs.csv`, so results can be compared across changes. Set `RUN_NOTE` to describe what changed before running.

In [8]:
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error

X = trainer[facility_types]
y = trainer["OverallQuality"]
groups = trainer["GeoID"]

xgb_params = dict(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=42,
)

# short description of what changed in this run; saved with the stats in model_runs.csv
RUN_NOTE = "infant mortality replaces premature deaths; new weights"

cv_pred = cross_val_predict(XGBRegressor(**xgb_params), X, y, groups=groups, cv=GroupKFold(n_splits=5))
cv_r2 = r2_score(y, cv_pred)
cv_mae = mean_absolute_error(y, cv_pred)
baseline_mae = mean_absolute_error(y, np.full(len(y), y.mean()))
# best R^2 any facilities-only model could reach: facility counts are the same in every period,
# so the most it can do is predict each district's average score
ceiling_r2 = r2_score(y, trainer.groupby("GeoID")["OverallQuality"].transform("mean"))

print(f"grouped 5-fold CV:  R^2 = {cv_r2:.3f}   MAE = {cv_mae:.4f}")
print(f"baseline (predict the mean): MAE = {baseline_mae:.4f}")
print(f"ceiling for a facilities-only model: R^2 = {ceiling_r2:.3f}")

# append this run's stats to the run log
import json, datetime, os
run = {
    "run_time": datetime.datetime.now().isoformat(timespec="seconds"),
    "note": RUN_NOTE,
    "cv_r2": round(cv_r2, 4),
    "cv_mae": round(cv_mae, 4),
    "baseline_mae": round(baseline_mae, 4),
    "ceiling_r2": round(ceiling_r2, 4),
    "n_rows": len(trainer),
    "n_districts": trainer["GeoID"].nunique(),
    "n_features": len(facility_types),
    "metrics": "; ".join(metric_cols),
    "weights": json.dumps(weights),
    "xgb_params": json.dumps(xgb_params),
}
runs_path = params_path + "model_runs.csv"
pd.DataFrame([run]).to_csv(runs_path, mode="a", header=not os.path.exists(runs_path), index=False)

model = XGBRegressor(**xgb_params)
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=facility_types).sort_values(ascending=False)
importances.head(10)

grouped 5-fold CV:  R^2 = 0.619   MAE = 0.1019
baseline (predict the mean): MAE = 0.1650
ceiling for a facilities-only model: R^2 = 0.991


TRANSPORTATION                   0.213741
HUMAN SERVICES                   0.134349
CAMPS                            0.104507
CHILD SERVICES AND WELFARE       0.083648
CULTURAL INSTITUTIONS            0.071608
YOUTH SERVICES                   0.056213
HISTORICAL SITES                 0.047734
DAY CARE AND PRE-KINDERGARTEN    0.047682
PARKS AND PLAZAS                 0.045080
LIBRARIES                        0.037634
dtype: float32

4. Model execution: for each (district, facility type) pair, use the model to calculate the predicted domain scores, then calculate the predicted domain percentiles and the Predicted Overall Percentile Lift.


For each district we predict a base score from its current facility counts, then add `ADD_N` facilities of one type and predict again. The lift is the difference. Tree models change their prediction only when a count crosses a split threshold, so adding a single facility often gives zero lift; raise `ADD_N` to ask "what if we built several of these?".

The percentile lift compares where the district's predicted score falls among all 59 districts' current predicted scores, before and after the change.

In [9]:
ADD_N = 1

base_X = facility_counts.set_index("GeoID")[facility_types]
base_pred = pd.Series(model.predict(base_X), index=base_X.index)

def percentile_among(values, reference):
    """share of reference values strictly below each value (0..1)"""
    ref = np.sort(reference)
    return np.searchsorted(ref, values, side="left") / (len(ref) - 1)

base_pct = percentile_among(base_pred.values, base_pred.values)

score_lift = pd.DataFrame(index=base_X.index, columns=facility_types, dtype=float)
pct_lift = pd.DataFrame(index=base_X.index, columns=facility_types, dtype=float)
for fac in facility_types:
    modified = base_X.copy()
    modified[fac] += ADD_N
    new_pred = model.predict(modified)
    score_lift[fac] = new_pred - base_pred.values
    pct_lift[fac] = percentile_among(new_pred, base_pred.values) - base_pct

print(f"share of (district, facility) pairs with non-zero lift: {(score_lift != 0).mean().mean():.1%}")
score_lift.round(4).head(10)

share of (district, facility) pairs with non-zero lift: 55.8%


,ADULT SERVICES,CAMPS,CHILD SERVICES AND WELFARE,"CITY AGENCY PARKING, MAINTENANCE, AND STORAGE",CULTURAL INSTITUTIONS,DAY CARE AND PRE-KINDERGARTEN,EMERGENCY SERVICES,HEALTH CARE,HIGHER EDUCATION,HISTORICAL SITES,...,OTHER PROPERTY,PARKS AND PLAZAS,PUBLIC SAFETY,SCHOOLS (K-12),SOLID WASTE,TELECOMMUNICATIONS,TRANSPORTATION,VOCATIONAL AND PROPRIETARY SCHOOLS,WATER AND WASTEWATER,YOUTH SERVICES
GeoID,,,,,,,,,,,,,,,,,,,,,
101,-0.0035,0.0030,0.0016,0.0000,0.0,0.0004,-0.0071,0.0001,0.0000,0.0,...,-0.0000,0.0,-0.0007,0.0000,0.0,-0.0056,0.0000,0.0000,-0.0005,0.0
102,0.0004,0.0012,-0.0685,-0.0028,0.0,0.0000,-0.0002,0.0000,0.0000,0.0,...,0.0059,0.0,0.0004,0.0000,0.0,-0.0115,0.0000,0.0000,0.0022,0.0
103,0.0022,0.0003,0.0000,0.0130,0.0,0.0097,0.0006,0.0000,0.0002,0.0,...,0.0000,0.0,0.0000,0.0002,0.0,0.0000,0.0000,0.0010,0.0123,0.0
104,-0.0052,0.0312,-0.0023,-0.0085,0.0,0.0001,-0.0039,0.0000,0.0019,0.0,...,0.0058,0.0,0.0007,0.0000,0.0,0.0000,0.0000,0.0008,0.0026,0.0
105,-0.0208,0.0000,0.0001,-0.0027,0.0,0.0000,-0.0039,0.0000,0.0000,0.0,...,0.0029,0.0,0.0000,0.0003,0.0,0.0000,0.0000,0.0000,0.0020,0.0
106,-0.0016,-0.0011,0.0000,0.0071,0.0,0.0006,0.0031,0.0000,0.0000,0.0,...,0.0000,0.0,-0.0003,0.0000,0.0,-0.0089,0.0000,-0.0011,-0.0004,0.0
107,0.0014,0.0000,-0.0885,0.0002,0.0,0.0000,0.0000,0.0000,0.0000,0.0,...,0.0000,0.0,0.0022,0.0000,0.0,-0.0115,0.0000,0.0002,-0.0001,0.0
108,-0.0011,0.0000,0.0000,0.0004,0.0,0.0000,-0.0004,0.0000,0.0000,0.0,...,0.0029,0.0,-0.0002,0.0000,0.0,-0.0115,0.0000,-0.0000,-0.0007,0.0
109,-0.0001,0.0000,0.0000,0.0000,0.0,0.0000,0.0010,0.0000,0.0000,0.0,...,0.0000,0.0,0.0007,0.0002,0.0,0.0000,0.0000,-0.0006,0.0000,0.0


5. Civil planning: for each district, compare the Predicted Overall Percentile Lift of every facility type to determine which facility type would most improve the quality of life for that district, and for the city as a whole, compare the highest Predicted Overall Percentile Lift from each district to determine which facility types should be built and in which districts they should be built.


In [ ]:
best_type = score_lift.idxmax(axis=1)
best_facilities = pd.DataFrame({
    "GeoID": score_lift.index,
    # facility type whose addition gives the biggest predicted increase in OverallQuality
    "recommended_facility": best_type.values,
    # model's OverallQuality (0-1) for the district as it is today
    "predicted_quality_now": base_pred.values,
    # model's OverallQuality after adding ADD_N of the recommended facility
    "predicted_quality_after": base_pred.values + score_lift.max(axis=1).values,
    # difference between the two (same 0-1 scale)
    "predicted_quality_gain": score_lift.max(axis=1).values,
    # how many percentile points the district would climb among all 59 districts (0-100 scale)
    "predicted_percentile_gain_pts": [100 * pct_lift.loc[g, f] for g, f in zip(best_type.index, best_type.values)],
})
# a district whose best gain is <= 0 gets no recommendation
best_facilities.loc[best_facilities["predicted_quality_gain"] <= 0, "recommended_facility"] = None

best_facilities = best_facilities.sort_values("predicted_quality_gain", ascending=False).reset_index(drop=True)
best_facilities.to_csv(facs_path + f"best_facilities_{run.get('run_time').replace(':', '.')}.csv", index=False)

print("City-wide: top 10 (district, facility) recommendations")
display(best_facilities.head(10))

print("How often each facility type is a district's best option:")
best_facilities["recommended_facility"].value_counts()

OSError: [Errno 22] Invalid argument: 'runs/facs/best_facilities_2026-11-24T14:49:58.csv'

6. Interactive output: display both the input data and our calculated results on a user-friendly interactive map to aid in interpretation.